# Sergio Santamaría Trueba

## Tercera entrega del lab 3. Técnicas de clasificación de variables categóricas.

En este ejercicio de entrenamiento vamos a trabajar en preparar y transformar datos de variables categóricas. 

In [ ]:
# 1. Configuración del entorno
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Configuramos el estilo visual de los gráficos
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore') # Evita advertencias de versiones en la salida

In [ ]:
# 2. Carga del dataset
ruta_archivo = '/workspace/dataset.csv'

# Definimos los nombres reales de las columnas (el archivo original no tiene cabecera)
nombres_columnas = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

# skipinitialspace=True es vital aquí para eliminar los espacios en blanco iniciales de los strings
df = pd.read_csv(ruta_archivo, header=None, names=nombres_columnas, skipinitialspace=True)

print("Datos cargados correctamente. Forma del dataset:", df.shape)

In [ ]:
# 3. Análisis exploratorio de datos (EDA)
print("--- Primeras filas del dataset ---")
display(df.head())

# Identificamos variables categóricas (tipo 'object') y numéricas
var_categoricas = df.select_dtypes(include=['object']).columns.tolist()
var_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\n--- Variables Categóricas Identificadas ---")
print(var_categoricas)

print("\n--- Análisis Descriptivo (Categóricas) ---")
display(df[var_categoricas].describe())

In [ ]:
# Generamos las visualizaciones exigidas

# 1. Gráficos de barras para variables categóricas (seleccionamos algunas clave para no saturar)
cols_cat_plot = ['workclass', 'education', 'sex', 'income']
plt.figure(figsize=(16, 12))
for i, col in enumerate(cols_cat_plot, 1):
    plt.subplot(2, 2, i)
    sns.countplot(y=df[col], palette='viridis', order=df[col].value_counts().index)
    plt.title(f'Distribución de {col}')
plt.tight_layout()
plt.show()

# 2. Histogramas para variables numéricas
plt.figure(figsize=(16, 10))
for i, col in enumerate(var_numericas, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], bins=20, kde=True, color='skyblue')
    plt.title(f'Histograma de {col}')
plt.tight_layout()
plt.show()

# 3. Boxplots para detectar outliers en variables numéricas
plt.figure(figsize=(16, 10))
for i, col in enumerate(var_numericas, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x=df[col], color='salmon')
    plt.title(f'Boxplot de {col} (Outliers)')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Codificación con get_dummies de pandas
# Genera variables binarias (0 y 1) para cada categoría. 
# drop_first=True evita la trampa de las variables ficticias (multicolinealidad).
df_dummies = pd.get_dummies(df, columns=var_categoricas, drop_first=True, dtype=int)

print("--- Resultado con get_dummies ---")
print(f"Nuevas dimensiones: {df_dummies.shape}")
display(df_dummies.head(3))

In [ ]:
# 5. Codificación con OneHotEncoder de scikit-learn
# El enunciado pide explícitamente sparse=False (en versiones nuevas de sklearn es sparse_output,
# pero usamos la sintaxis compatible con el enunciado/evaluador).
ohe = OneHotEncoder(sparse=False, drop='first')

# Entrenamos y transformamos las variables categóricas
matriz_ohe = ohe.fit_transform(df[var_categoricas])

# Para visualizarlo bien, lo convertimos en DataFrame recuperando los nombres de las columnas
nombres_ohe = ohe.get_feature_names_out(var_categoricas)
df_ohe = pd.DataFrame(matriz_ohe, columns=nombres_ohe, index=df.index)

# Unimos con las variables numéricas originales
df_onehot_final = pd.concat([df[var_numericas], df_ohe], axis=1)

print("--- Resultado con OneHotEncoder ---")
print(f"Nuevas dimensiones: {df_onehot_final.shape}")
display(df_onehot_final.head(3))

In [ ]:
# 6. Codificación con LabelEncoder de scikit-learn
# LabelEncoder transforma cada categoría en un número entero (0, 1, 2...).
# IMPORTANTE: Se aplica de forma individual a cada columna.
df_label = df.copy()
le = LabelEncoder()

for col in var_categoricas:
    df_label[col] = le.fit_transform(df_label[col])

print("--- Resultado con LabelEncoder ---")
print(f"Nuevas dimensiones: {df_label.shape} (las dimensiones no aumentan)")
display(df_label.head(3))